# <center>Method Chaining and Functional Pandas

Biggest difference between a beginner and a senior data scientist\
**A beginner writes code like this:**\
df1 = df[df["Salary"] > 50000]\
df2 = df1.drop_duplicates()\
df3 = df2.sort_values("Salary")\
df4 = df3.reset_index(drop=True)\
df5 = df4.assign(Bonus=df4["Salary"] * 0.1)\
It works.

But imagine doing this for 50 steps.\
It becomes unreadable.\
**A professional writes:**\
(\
    df\
    .query("Salary > 50000")\
    .drop_duplicates()\
    .sort_values("Salary")\
    .reset_index(drop=True)\
    .assign(Bonus=lambda x: x["Salary"] * 0.1)\
)

One clean pipeline.\
That's **Method Chaining**

Think of a DataFrame flowing through a pipeline:\
Raw Data --> Filter --> Clean --> Sort --> Create Columns --> Aggregate --> Output\
Every operation returns another DataFrame.\
That means the next operation can immediately continue

**Dataset** 

In [6]:
import pandas as pd
df = pd.DataFrame({
    "Name":["Ali","Sara","Ahmed","Sara","Usman"],
    "Department":["HR","IT","IT","IT","Finance"],
    "Salary":[50000,70000,90000,70000,60000],
    "Experience":[2,5,8,5,3]
})

In [2]:
df

,Name,Department,Salary,Experience
0,Ali,HR,50000,2
1,Sara,IT,70000,5
2,Ahmed,IT,90000,8
3,Sara,IT,70000,5
4,Usman,Finance,60000,3


**1. Method Chaining Basics**

Instead of:\
df = df.drop_duplicates()\
df = df.sort_values("Salary")\
df = df.reset_index(drop=True)

In [7]:
(
    df
    .drop_duplicates()
    .sort_values("Salary")
    .reset_index(drop=True)
) 

,Name,Department,Salary,Experience
0,Ali,HR,50000,2
1,Usman,Finance,60000,3
2,Sara,IT,70000,5
3,Ahmed,IT,90000,8


Without Parentheses the code becomes ugly.\
With Parentheses its readable

**2. assign()**\
Used to create a new column

Normally:\
df["Bonus"] = df["Salary"] * 0.1\
Method Chaining:

In [12]:
df = (
    df
    .assign(
        Bonus=lambda x:
        x["Salary"]*0.1,
        Tax=lambda x:
        x["Bonus"]*0.2
    )
)

In [13]:
df

,Name,Department,Salary,Experience,Bonus,Tax
0,Ali,HR,50000,2,5000.0,1000.0
1,Sara,IT,70000,5,7000.0,1400.0
2,Ahmed,IT,90000,8,9000.0,1800.0
3,Sara,IT,70000,5,7000.0,1400.0
4,Usman,Finance,60000,3,6000.0,1200.0


Inside assign():\
Lambda x means Current DataFrame after previous operations.

**3. query()**

In [14]:
(
    df
    .query(
        "Salary > 60000 and Department == 'IT'"
    )
)

,Name,Department,Salary,Experience,Bonus,Tax
1,Sara,IT,70000,5,7000.0,1400.0
2,Ahmed,IT,90000,8,9000.0,1800.0
3,Sara,IT,70000,5,7000.0,1400.0


Cleaner, more readable and fits naturally inside chains.

**4. pipe()**\
Used for custom functions

In [15]:
def add_bonus(df):
    df["Bonus"] = df["Salary"] * 0.1
    return df
(
    df
    .pipe(add_bonus)
)

,Name,Department,Salary,Experience,Bonus,Tax
0,Ali,HR,50000,2,5000.0,1000.0
1,Sara,IT,70000,5,7000.0,1400.0
2,Ahmed,IT,90000,8,9000.0,1800.0
3,Sara,IT,70000,5,7000.0,1400.0
4,Usman,Finance,60000,3,6000.0,1200.0


Fits into the chain unlike normal function call

**5. sort_values()**

In [16]:
(
    df
    .query(
        "Salary > 50000"
    )
    .sort_values(
        "Salary",
        ascending = False
    )
)

,Name,Department,Salary,Experience,Bonus,Tax
2,Ahmed,IT,90000,8,9000.0,1800.0
1,Sara,IT,70000,5,7000.0,1400.0
3,Sara,IT,70000,5,7000.0,1400.0
4,Usman,Finance,60000,3,6000.0,1200.0


**6. rename()**\
Renaming inside pipelines

In [17]:
(
    df
    .rename(
        columns={
            "Salary":"Income"
        }
    )
)

,Name,Department,Income,Experience,Bonus,Tax
0,Ali,HR,50000,2,5000.0,1000.0
1,Sara,IT,70000,5,7000.0,1400.0
2,Ahmed,IT,90000,8,9000.0,1800.0
3,Sara,IT,70000,5,7000.0,1400.0
4,Usman,Finance,60000,3,6000.0,1200.0


**7. Combining Everything**

Remove duplicates\
Keep salary>60000\
Add bonus\
Add tax\
Sort by Salary\
Reset index

In [19]:
import pandas as pd
df = pd.DataFrame({
    "Name":["Ali","Sara","Ahmed","Sara","Usman"],
    "Department":["HR","IT","IT","IT","Finance"],
    "Salary":[50000,70000,90000,70000,60000],
    "Experience":[2,5,8,5,3]
})

def add_bonus(df):
    df["Bonus"] = df["Salary"] * 0.1
    return df
def add_tax(df):
    df["Tax"] = df["Salary"] * 0.05
    return df

(
    df
    .drop_duplicates()
    .query("Salary > 60000")
    .pipe(add_bonus)
    .pipe(add_tax)
    .sort_values(
        "Salary",
        ascending = False
    )
    .reset_index(drop=True)
)

,Name,Department,Salary,Experience,Bonus,Tax
0,Ahmed,IT,90000,8,9000.0,4500.0
1,Sara,IT,70000,5,7000.0,3500.0


**8. Writing Reusable Pipelines**\
Instead of copying the same cleaning steps across notebooks:

In [20]:
def clean_employee_data(df):
    return (
        df
        .drop_duplicates()
        .dropna()
        .query("Salary > 0")
        .assign(
            Bonus=lambda x:
            x["Salary"]*0.1
        )
        .reset_index(drop=True)
    )

clean_df = clean_employee_data(df)

In [21]:
clean_df

,Name,Department,Salary,Experience,Bonus
0,Ali,HR,50000,2,5000.0
1,Sara,IT,70000,5,7000.0
2,Ahmed,IT,90000,8,9000.0
3,Usman,Finance,60000,3,6000.0


### Method Chaining Pipeline

Read data --> Remove Duplicates --> Remove missing values --> Filter --> Create Columns --> Sort --> Aggregate --> Export

### Professional Rules

Rule 1

Prefer one clean pipeline over many temporary DataFrames.

Rule 2

Use assign() instead of creating columns outside the chain.

Rule 3

Use pipe() for custom logic.

Rule 4

Keep each step on its own line.

Rule 5

Each line should do one thing.